In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

# Crear el esquema 'silver' si no existe and added a new comment here
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_schema = StructType([
    StructField("ticket_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_sales_amount", DoubleType(), True),
    StructField("transaction_time", TimestampType(), True),
    StructField("processed_at", TimestampType(), True)
])

from pyspark.sql.functions import col, current_timestamp, to_timestamp

# 1. Leer de la tabla Bronze (Streaming)
df_bronze = spark.readStream.table("workspace.default.bronze_sales_stream")

# 2. Transformaciones de Capa Silver
df_silver = (df_bronze
    # Aseguramos tipos de datos correctos
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("transaction_time", to_timestamp(col("timestamp")))
    
    # Lógica de Negocio: Cálculo de Monto Total
    .withColumn("total_sales_amount", col("quantity") * col("unit_price"))
    
    # Limpieza: Filtramos registros que no tengan sentido para el negocio
    .filter(col("quantity") > 0)
    .filter(col("unit_price").isNotNull())
    
    # Metadatos de auditoría para Silver
    .withColumn("processed_at", current_timestamp())
    .select(
        "ticket_id", "store_id", "sku", "quantity", 
        "unit_price", "total_sales_amount", "transaction_time", "processed_at"
    )
)

# 3. Escribir en la tabla Silver en el nuevo esquema 'silver'
checkpoint_silver = "/Volumes/workspace/default/tmp_landing/_checkpoints/sales_silver"

query_silver = (df_silver.writeStream
    .option("checkpointLocation", checkpoint_silver)
    .trigger(availableNow=True)
    .outputMode("append")
    .toTable("silver.silver_sales")
)